# Glutamate response analysis run

Use this notebook to load one session, run activation/tuning/sequence analyses, and save standardized output tables.

In [1]:
import os
import glob
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from vip_slap2_analysis.glutamate.analysis import (
    GlutamateAnalysisConfig,
    resolve_glutamate_analysis_paths,
    run_glutamate_analysis,
)

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
%load_ext autoreload
%autoreload 2

In [10]:
target_mice = [826033]

In [ ]:
registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

In [9]:
assets[0]

SessionAssets(session_id='826033_2026-02-17_13-13-55', subject_id=826033, session_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/826033_2026-02-17_13-13-55'), summary_mat=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/826033_2026-02-17_13-13-55/826033_2026-02-17_13-13-55_slap2_2026-02-17_13-13-55/source_extraction/ExperimentSummary/SummaryLoCo-260218-072719.mat'), bonsai_event_log_csv=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/826033_2026-02-17_13-13-55/826033_2026-02-17_13-13-55/behavior/VCO1_Behavior.harp/bonsai_event_log.csv'), harp_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/826033_2026-02-17_13-13-55/826033_2026-02-17_13-13-55/behavior/VCO1_Behavior.harp'), photodiode_pkl=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/826033_2026-02-17

In [6]:
for asset in assets[1:]:
    session_root = asset.session_dir
    paths = resolve_glutamate_analysis_paths(session_root)
    print(paths)
    
    config = GlutamateAnalysisConfig(
    alpha=0.05,
    min_events_activation=8,
    min_events_tuning_per_image=5,
    min_images_for_tuning=4,
    min_images_for_sequence=4,
    min_positions_for_sequence=3,
    n_shuffles_tuning=1000,
    random_seed=0,
    )
    config
    
    results = run_glutamate_analysis(session_root, config=config)
    {k: v.shape for k, v in results.items() if hasattr(v, 'shape')}
    
    activation_summary = results['activation_summary_table']
    tuning_summary = results['tuning_summary_table']
    sequence_summary = results['sequence_summary_table']

    display(activation_summary.head())
    display(tuning_summary.head())
    display(sequence_summary.head())
    
    output_dir = resolve_glutamate_analysis_paths(session_root).output_dir
    print(output_dir)
    print(sorted(p.name for p in output_dir.iterdir()))

GlutamateAnalysisPaths(single_trial_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_single_trial_df.npz'), mean_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_mean_df.npz'), sequence_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_sequence_df.npz'), output_dir=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-25_803496/analysis/derived/glutamate/glutamate_analysis'))


,session_id,subject_id,dmd,synapse_id,stimulus_family,n_events,median_delta_auc,mean_delta_auc,effect_direction,p_value,response_class,q_value_within_synapse
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0000,image,2277,-20.015213,-23.085767,none,3.356907e-02,no_change,1.007072e-01
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0001,image,2277,11.062568,10.113008,none,1.885622e-01,no_change,3.431282e-01
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,image,2277,62.247024,75.345806,up,1.833899e-10,activated,5.501698e-10
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0003,image,2277,51.070708,94.594795,up,5.779422e-15,activated,1.733827e-14
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0004,image,2277,-94.736861,-159.029189,down,1.601894e-34,deactivated,4.805682e-34


,session_id,subject_id,dmd,synapse_id,n_image_trials,n_images_tested,fve_image,p_shuffle_fve,p_kw,preferred_image,preferred_mean,preferred_median,preferred_vs_rest_effect,preferred_vs_next_effect,is_tuned,q_shuffle_fve,q_kw
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,2277,7,0.016428,0.000999,6.161776e-06,imk01643,209.877494,194.113103,148.201225,79.053404,False,0.001556,1.247760e-05
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0003,2277,7,0.029130,0.000999,2.518969e-11,imk01378,186.991674,110.479186,69.505918,6.580640,False,0.001556,7.847557e-11
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0005,2277,7,0.090994,0.000999,1.177659e-48,imk01643,622.228322,622.815466,348.406914,133.607561,True,0.001556,1.907808e-47
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0006,2277,7,0.007872,0.009990,5.606826e-03,imk01220,128.009940,143.542635,117.889387,40.459149,False,0.013051,7.569215e-03
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0007,2277,7,0.017145,0.000999,1.310140e-05,imk00895,117.075662,97.514651,58.983192,8.945764,False,0.001556,2.526698e-05


,session_id,subject_id,dmd,synapse_id,n_images_with_sequences,median_seq_slope,median_adaptation_index,median_r0,median_rlast,median_rterminal,median_terminal_minus_last,median_early_minus_late,seq_p,sequence_class,seq_q
0,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0002,7,-8.076821,1.000000,92.015880,-276.286065,102.560630,305.798560,360.275090,0.21875,stable,0.454327
1,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0003,7,-0.722605,0.858060,44.223449,-122.997100,70.886819,97.337593,311.847098,0.68750,stable,0.807065
2,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0005,7,-7.898842,1.000000,349.948409,-172.897076,448.438109,690.713444,217.207262,0.03125,stable,0.158203
3,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0006,7,-1.355001,0.898264,75.070938,-176.820424,22.233419,199.150761,120.485039,0.68750,stable,0.807065
4,803496_2025-07-25_13-02-10,803496,DMD1,DMD1_syn0007,7,-5.214190,1.000000,64.219228,-54.981275,47.567511,173.151400,105.636095,0.21875,stable,0.454327


\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496\analysis\derived\glutamate\glutamate_analysis
['activation_event_table.csv', 'activation_event_table.parquet', 'activation_summary_table.csv', 'activation_summary_table.parquet', 'glutamate_analysis_metadata.json', 'sequence_per_image_table.csv', 'sequence_per_image_table.parquet', 'sequence_summary_table.csv', 'sequence_summary_table.parquet', 'tuning_per_image_table.csv', 'tuning_per_image_table.parquet', 'tuning_summary_table.csv', 'tuning_summary_table.parquet']
GlutamateAnalysisPaths(single_trial_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-28_803496/analysis/derived/glutamate/glutamate_single_trial_df.npz'), mean_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f/803496/2025-07-28_803496/analysis/derived/glutamate/glutamate_mean_df.npz'), sequence_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_sy

...

,session_id,subject_id,dmd,synapse_id,stimulus_family,n_events,median_delta_auc,mean_delta_auc,effect_direction,p_value,response_class,q_value_within_synapse
0,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0000,image,2280,-11.570199,-31.118654,down,5.504283e-13,deactivated,1.651285e-12
1,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0001,image,2280,-4.074806,-35.771417,down,8.290271e-14,deactivated,2.487081e-13
2,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0002,image,2280,0.000000,-9.559323,none,7.910649e-02,no_change,2.373195e-01
3,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0003,image,2280,0.000000,1.325095,none,3.121878e-01,no_change,3.121878e-01
4,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0004,image,2280,51.802938,137.968067,up,2.687476e-73,activated,8.062429e-73


,session_id,subject_id,dmd,synapse_id,n_image_trials,n_images_tested,fve_image,p_shuffle_fve,p_kw,preferred_image,preferred_mean,preferred_median,preferred_vs_rest_effect,preferred_vs_next_effect,is_tuned,q_shuffle_fve,q_kw
0,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0004,2280,7,0.409365,0.000999,3.632036e-97,216066,673.759422,647.187659,616.259942,571.640344,True,0.001213,3.087231e-96
1,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0006,2280,7,0.004701,0.091908,5.476208e-02,216066,43.265211,0.598394,-13.363881,11.132018,False,0.097652,5.818471e-02
2,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0008,2280,7,0.046888,0.000999,9.752102e-12,imk01306,144.434773,77.334082,77.334082,81.535337,False,0.001213,1.507143e-11
3,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0010,2280,7,0.170702,0.000999,6.425625e-74,41006,527.999481,564.862955,326.772614,101.893456,True,0.001213,3.641188e-73
4,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0012,2280,7,0.129902,0.000999,2.787110e-37,imk01333,346.897380,230.709164,204.401276,197.530177,True,0.001213,9.476175e-37


,session_id,subject_id,dmd,synapse_id,n_images_with_sequences,median_seq_slope,median_adaptation_index,median_r0,median_rlast,median_rterminal,median_terminal_minus_last,median_early_minus_late,seq_p,sequence_class,seq_q
0,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0004,7,-3.614721,-0.173060,68.433730,97.076995,156.356137,56.802020,-62.179567,0.15625,stable,0.664062
1,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0006,7,-0.010323,1.000000,25.345669,-171.626236,30.831925,88.631962,-36.792032,1.00000,stable,1.000000
2,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0008,7,0.383531,-0.619604,62.329301,292.975370,45.701775,-192.205607,-133.069137,0.81250,stable,1.000000
3,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0010,7,-5.862908,0.006598,284.467885,363.335288,381.951057,33.039563,-18.039001,0.03125,stable,0.265625
4,803121_2025-11-06_12-12-23,803121,DMD1,DMD1_syn0012,7,-0.608527,0.337900,72.100396,78.941290,120.769246,60.352170,-18.425578,0.93750,stable,1.000000


\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803121\2025-11-06_803121\analysis\derived\glutamate\glutamate_analysis
['activation_event_table.csv', 'activation_event_table.parquet', 'activation_summary_table.csv', 'activation_summary_table.parquet', 'glutamate_analysis_metadata.json', 'sequence_per_image_table.csv', 'sequence_per_image_table.parquet', 'sequence_summary_table.csv', 'sequence_summary_table.parquet', 'tuning_per_image_table.csv', 'tuning_per_image_table.parquet', 'tuning_summary_table.csv', 'tuning_summary_table.parquet']
GlutamateAnalysisPaths(single_trial_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/826033_2026-02-17_13-13-55/analysis/derived/glutamate/glutamate_single_trial_df.npz'), mean_npz=WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/826033_2026-02-17_13-13-55/analysis/derived/glutamate/glutamate_mean_df.npz'), sequence_npz=WindowsPath('//allen/

FileNotFoundError: NPZ not found: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\826033_2026-02-17_13-13-55\analysis\derived\glutamate\glutamate_single_trial_df.npz